# Vancouver Airbnb — Feature Engineering

Reads `datasets/final_dataset.csv` (88 columns) and adds **14 new meaningful features**,
producing `datasets/final_dataset_features.csv` with **102 columns**.

All original columns and values are preserved unchanged.

## New features overview

| # | Feature | Group | Description |
|---|---|---|---|
| 89 | `desc_word_count` | Description NLP | Word count of listing description |
| 90 | `desc_char_count` | Description NLP | Character count of listing description |
| 91 | `desc_sentiment_compound` | Description NLP | Overall sentiment score (VADER, −1 to +1) |
| 92 | `desc_sentiment_positive` | Description NLP | Proportion of positive-sentiment words |
| 93 | `desc_readability_fk_grade` | Description NLP | Flesch-Kincaid reading grade level |
| 94 | `desc_exclamation_density` | Description NLP | Exclamation marks per word (enthusiasm signal) |
| 95 | `host_about_word_count` | Host Profile NLP | Word count of host self-description |
| 96 | `host_about_sentiment_compound` | Host Profile NLP | Sentiment tone of host self-description |
| 97 | `amenity_count_total` | Amenities | Total number of amenities listed |
| 98 | `amenity_has_workspace` | Amenities | Binary: dedicated workspace or desk available |
| 99 | `amenity_has_parking` | Amenities | Binary: any parking option available |
| 100 | `amenity_has_pool_gym` | Amenities | Binary: pool, gym or fitness facility available |
| 101 | `host_tenure_days` | Host Experience | Days from host_since to the snapshot date |
| 102 | `price_numeric` | Price | Numeric price (stripped of $ and commas) |

---
**Requirements:**
```
pip install pandas vaderSentiment textstat
```

## 1 — Imports

In [ ]:
import os
import re
import ast
import numpy as np
import pandas as pd
from IPython.display import display
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import textstat

print("All libraries loaded successfully.")

## 2 — Configuration

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath("__file__"))
INPUT_PATH = os.path.join(SCRIPT_DIR, "datasets", "final_dataset.csv")
OUTPUT_PATH = os.path.join(SCRIPT_DIR, "datasets", "final_dataset_features.csv")

print(f"Input  : {INPUT_PATH}")
print(f"Output : {OUTPUT_PATH}")

## 3 — Load dataset

In [ ]:
df = pd.read_csv(INPUT_PATH, low_memory=False)

print(f"Loaded : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Expected 88 columns — OK: {df.shape[1] == 88}")

## 4 — Helper functions

In [ ]:
analyzer = SentimentIntensityAnalyzer()


def clean_text(val):
    """Strip HTML tags and return plain text; return empty string for NaN."""
    if pd.isna(val):
        return ""
    return re.sub(r"<[^>]+>", " ", str(val)).strip()


def parse_amenities(val):
    """Parse the JSON-like amenities string into a list of lowercase strings."""
    try:
        return [x.lower() for x in ast.literal_eval(val)]
    except Exception:
        return []


print("Helper functions defined.")

## 5 — Group A: Listing description NLP (6 features)

These features capture *how* hosts write their listing description —
the core signal for the research question.

In [ ]:
print("Computing description NLP features...")

desc_clean = df["description"].apply(clean_text)

# 89 — word count
df["desc_word_count"] = desc_clean.apply(lambda t: len(t.split()) if t else 0)

# 90 — character count
df["desc_char_count"] = desc_clean.apply(len)

# 91 & 92 — VADER sentiment
vader_scores = desc_clean.apply(
    lambda t: analyzer.polarity_scores(t) if t else {"compound": 0.0, "pos": 0.0}
)
df["desc_sentiment_compound"] = vader_scores.apply(lambda x: x["compound"])
df["desc_sentiment_positive"] = vader_scores.apply(lambda x: x["pos"])

# 93 — Flesch-Kincaid readability grade
# NaN when fewer than 5 words (not enough text to measure reliably)
df["desc_readability_fk_grade"] = desc_clean.apply(
    lambda t: textstat.flesch_kincaid_grade(t) if len(t.split()) >= 5 else np.nan
)

# 94 — exclamation mark density (exclamations per word)
df["desc_exclamation_density"] = desc_clean.apply(
    lambda t: t.count("!") / len(t.split()) if len(t.split()) > 0 else 0.0
)

print("  desc_word_count            done")
print("  desc_char_count            done")
print("  desc_sentiment_compound    done")
print("  desc_sentiment_positive    done")
print("  desc_readability_fk_grade  done")
print("  desc_exclamation_density   done")

## 6 — Group B: Host profile NLP (2 features)

Directly addresses the research question — does the *tone and length*
of how a host describes themselves affect occupancy?

In [ ]:
print("Computing host_about NLP features...")

about_clean = df["host_about"].apply(clean_text)

# 95 — word count (0 when host_about is missing)
df["host_about_word_count"] = about_clean.apply(lambda t: len(t.split()) if t else 0)

# 96 — VADER sentiment (NaN when host_about is missing)
df["host_about_sentiment_compound"] = about_clean.apply(
    lambda t: analyzer.polarity_scores(t)["compound"] if t else np.nan
)

print("  host_about_word_count            done")
print("  host_about_sentiment_compound    done")

## 7 — Group C: Amenity features (4 features)

Parsed from the raw JSON-like `amenities` string — capturing listing
quality signals that go beyond a simple count.

In [ ]:
print("Computing amenity features...")

amenities_list = df["amenities"].apply(parse_amenities)

# 97 — total amenity count
df["amenity_count_total"] = amenities_list.apply(len)

# 98 — has dedicated workspace or desk
df["amenity_has_workspace"] = amenities_list.apply(
    lambda a: 1 if any("workspace" in x or "desk" in x for x in a) else 0
).astype(int)

# 99 — has any parking option
df["amenity_has_parking"] = amenities_list.apply(
    lambda a: 1 if any("parking" in x for x in a) else 0
).astype(int)

# 100 — has pool, gym, or fitness facility
df["amenity_has_pool_gym"] = amenities_list.apply(
    lambda a: (
        1
        if any(keyword in x for x in a for keyword in ("pool", "gym", "fitness", "exercise"))
        else 0
    )
).astype(int)

print("  amenity_count_total     done")
print("  amenity_has_workspace   done")
print("  amenity_has_parking     done")
print("  amenity_has_pool_gym    done")

## 8 — Group D: Host experience (1 feature)

In [ ]:
print("Computing host tenure...")

# 101 — days from host_since to the snapshot date
# Captures host experience at the exact moment of each observation
snapshot_dt = pd.to_datetime(df["snapshot_date"], errors="coerce")
host_since_dt = pd.to_datetime(df["host_since"], errors="coerce")

df["host_tenure_days"] = (snapshot_dt - host_since_dt).dt.days

print("  host_tenure_days    done")

## 9 — Group E: Numeric price (1 feature)

In [ ]:
print("Computing numeric price...")

# 102 — strip $ and commas from the raw price string
df["price_numeric"] = df["price"].str.replace(r"[$,]", "", regex=True).astype(float)

print("  price_numeric    done")

## 10 — Verify final shape and new feature summary

In [ ]:
NEW_FEATURES = [
    "desc_word_count",
    "desc_char_count",
    "desc_sentiment_compound",
    "desc_sentiment_positive",
    "desc_readability_fk_grade",
    "desc_exclamation_density",
    "host_about_word_count",
    "host_about_sentiment_compound",
    "amenity_count_total",
    "amenity_has_workspace",
    "amenity_has_parking",
    "amenity_has_pool_gym",
    "host_tenure_days",
    "price_numeric",
]

print("Shape check:")
print("  Original columns : 88")
print(f"  New features     : {len(NEW_FEATURES)}")
print(f"  Final columns    : {df.shape[1]}  (expected 102)")
print(f"  Total rows       : {df.shape[0]:,}")
print(f"  Shape correct    : {df.shape == (43312, 102)}")
print()

# Summary table of new features
summary_rows = []
for col in NEW_FEATURES:
    summary_rows.append(
        {
            "Feature": col,
            "Non-null": df[col].notna().sum(),
            "Null (NaN)": df[col].isna().sum(),
            "Mean": round(df[col].mean(), 4) if df[col].dtype != object else "-",
            "Min": round(df[col].min(), 4) if df[col].dtype != object else "-",
            "Max": round(df[col].max(), 4) if df[col].dtype != object else "-",
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

## 11 — Confirm original data is unchanged

In [ ]:
original_df = pd.read_csv(INPUT_PATH, low_memory=False)
orig_cols = list(original_df.columns)

diff_cols = []
for col in orig_cols:
    o = original_df[col].reset_index(drop=True)
    m = df[col].reset_index(drop=True)
    if not ((o == m) | (o.isna() & m.isna())).all():
        diff_cols.append(col)

print(f"Original columns checked : {len(orig_cols)}")
print(
    f"Columns with any change  : {len(diff_cols)}  {'(none — all original data intact ✓)' if not diff_cols else diff_cols}"
)

## 12 — Save to `datasets/final_dataset_features.csv`

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved : {OUTPUT_PATH}")
print(f"Size  : {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.1f} MB")
print(f"Shape : {df.shape[0]:,} rows x {df.shape[1]} columns")